**Note**: If you haven’t done any training and you would like to test a pre-trained model, please refer to the notebook **test_with_pretrained.ipynb**!

## 3.a Test Using Last Trained Model

We are finally ready to test our trained model!

We will initially import the packages and define utils as in previous notebooks, then we will test the model.

When choosing which model to use, we recommend checking the intermediate outputs saved during training (see section **2.2.4 Training Definition** in **training.ipynb**) and use then the respective model.


### 3.1 Import Packages

In [2]:
import os
import sys

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import torch
import numpy as np
from model import  Self_net_architecture
import tifffile
import threading
import napari
import cv2
import matplotlib.pyplot as plt

### 3.2 Utils

We are defining several functions. We meet again the reslicing and in this case we need to perform an upsampling.

As we mentioned in the first notebook **data_generation.ipynb**, one of the crucial step is the downsampling of the XY plane images.

What we would like to do here is instead to upsample our volume. We will perform an interpolation on the stack and then correcting this process through our trained model. This upsample process is involving again a specific scale, that is the same that we used to downsample the XY images, i.e. the axial step divided by the pixel size.

This scale depends on your specific Z-stack and it is recommended to adjust it depending on your specific dataset!

This process will be done for both XZ and YZ planes. After applying the network the images along the x and y direction they will be mediate to reconstruct the entire stack. The reconstruction on XZ and YZ will be synchronized through threads. 

**Note**: the network will also work imposing a unitary size, i.e. `z_res = 1` and `x_res = 1`, and in this case you will get a z-stack in output that will have exactly the same dimensions of the z-stack in input.

#### 3.2.1 Reslicing

In [3]:
def reslice(img,position,x_res,z_res):
    scale=z_res/x_res
    z,y,x=img.shape
    if position=='xz':
        reslice_img=np.transpose(img,[1,0,2])

        scale_img=np.zeros((y,round(z*scale),x),dtype=np.uint16)
        for i in range(y):
            scale_img[i]=cv2.resize(reslice_img[i],(x,round(z*scale)),interpolation=cv2.INTER_CUBIC)

    else:
        reslice_img=np.transpose(img,[2,0,1])
        scale_img = np.zeros((x, round(z * scale), y), dtype=np.uint16)
        for i in range(x):
            scale_img[i] = cv2.resize(reslice_img[i], (y,round(z * scale)), interpolation=cv2.INTER_CUBIC)

    return scale_img

#### 3.2.2 Model Eval

In [4]:
def output_img(deblur_net,device,min_v,max_v,write_stack,raw_img):
    deblur_net.eval()
    device_cpu = torch.device('cpu')

    batch_size=3

    z_shape=raw_img.shape[0]

    idx = z_shape// batch_size

    res=z_shape-idx*batch_size

    input_img = (raw_img.astype(np.float32) - min_v) / (max_v - min_v)

    input_img[input_img> 1] = 1
    input_img[input_img < 0] = 0

    input_img = np.expand_dims(input_img, axis=1)
    input_tensor = torch.from_numpy(input_img)

    for ii in range(idx):
        with torch.no_grad():
            test_tensor=input_tensor[ii * batch_size:(ii + 1) * batch_size].to(device)
            net_output = deblur_net(test_tensor)
            print('{}/{}'.format((ii + 1) * batch_size, z_shape))

        net_output = net_output.squeeze_(1).to(device_cpu).numpy()
        net_output = net_output * (max_v - min_v) + min_v
        net_output = np.clip(net_output, 0, max_v).astype(np.uint16)

        write_stack[ii * batch_size:(ii + 1) * batch_size] = net_output


    if res!=0:
        test_tensor = input_tensor[idx * batch_size:].to(device)
        with torch.no_grad():
            net_output = deblur_net(test_tensor)
            print('{}/{}'.format(z_shape, z_shape))

        net_output = net_output.squeeze_(1).to(device_cpu).numpy()
        net_output = net_output * (max_v - min_v) + min_v
        net_output = np.clip(net_output, 0, max_v).astype(np.uint16)

        write_stack[idx * batch_size:] = net_output

#### 3.2.3 Thead creation

In [5]:
class myThread (threading.Thread):
    def __init__(self, threadID, name,deblur_net,device,min_v,max_v,write_stack,raw_img):
        threading.Thread.__init__(self)
        self.threadID = threadID
        self.name = name
        self.deblur_net=deblur_net
        self.device=device
        self.min_v=min_v
        self.max_v=max_v
        self.raw_img=raw_img
        self.write_stack=write_stack

    def run(self):
        print ("start threading：" + self.name)
        output_img(self.deblur_net, self.device, self.min_v,self.max_v,self.write_stack,self.raw_img)
        print ("quit threading：" + self.name)

#### 3.2.4 Upsampling

In [6]:
def upsample_block(raw_img,x_res,z_res,deblur_netA,deblur_netB, min_v, max_v):

    device1=torch.device('cuda:0')
    device2=torch.device('cuda:0')


    xz_img=reslice(raw_img,'xz',x_res,z_res)
    yz_img=reslice(raw_img,'yz',x_res,z_res)

    
    print(xz_img.shape,yz_img.shape)


    out_xz_img=np.zeros_like(xz_img,dtype=np.uint16)
    out_yz_img=np.zeros_like(yz_img,dtype=np.uint16)

    thread1=myThread(1,'thread:1',deblur_netA,device1,min_v,max_v,out_xz_img,xz_img)
    thread2=myThread(2,'thread:2',deblur_netB,device2,min_v,max_v,out_yz_img,yz_img)

    thread1.start()
    thread2.start()
    thread1.join()
    thread2.join()

    re_out_xz=np.transpose(out_xz_img,[1,0,2])
    re_out_yz=np.transpose(out_yz_img,[1,2,0])

    fusion_stack=re_out_xz/2+re_out_yz/2
    fusion_stack=np.array(fusion_stack,dtype=np.uint16)


    return fusion_stack, xz_img, yz_img

### 3.3 Model Test

In [8]:
model_path = parent_dir + '/checkpoint' + '/saved_models' + '/deblur_net_21_4000.pkl'  # Change this to load different trained models for testing.

save_directory = parent_dir + '/results'
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

input_img_path = parent_dir + '/first_channel_mahe_f16.tif'  # Change this to load different input images for testing.

In [9]:
device1 = torch.device('cuda:0')
device2 = torch.device('cuda:0') #if you have only one GPU, set both device1 and device2 to the same GPU. If you have two GPUs, set them to different GPUs.

deblur_net_A = Self_net_architecture.define_G(input_nc=1, output_nc=1, ngf=64, netG='deblur_net', device=device1,use_dropout=False,norm='instance')
deblur_net_A.load_state_dict(torch.load(model_path,map_location={'cuda:1':'cuda:0'}))

deblur_net_B = Self_net_architecture.define_G(input_nc=1, output_nc=1, ngf=64, netG='deblur_net', device=device2,use_dropout=False,norm='instance')
deblur_net_B.load_state_dict(torch.load(model_path))

min_v = 0
max_v = 255

raw_img=tifffile.imread(input_img_path)

x_res  = 0.275
z_res = 0.5

fusion_stack, xz_img, yz_img=upsample_block(raw_img,x_res,z_res,deblur_net_A,deblur_net_B,min_v,max_v)

initialize network with normal
initialize network with normal
(850, 798, 850) (850, 798, 850)
start threading：thread:1
start threading：thread:2
3/850
3/850
6/850
6/850
9/850
9/850
12/850
12/850
15/850
15/850
18/850
18/850
21/850
21/850
24/850
24/850
27/850
27/850
30/850
30/850
33/850
33/850
36/850
36/850
39/850
39/850
42/850
42/850
45/850
45/850
48/850
48/850
51/850
51/850
54/850
54/850
57/850
57/850
60/850
60/850
63/850
63/850
66/850
66/850
69/850
69/850
72/850
72/850
75/850
75/850
78/850
78/850
81/850
81/850
84/850
84/850
87/850
87/850
90/850
90/850
93/850
93/850
96/850
96/850
99/850
99/850
102/850
102/850
105/850
105/850
108/850
108/850
111/850
111/850
114/850
114/850
117/850
117/850
120/850
120/850
123/850
123/850
126/850
126/850
129/850
129/850
132/850
132/850
135/850
135/850
138/850
138/850
141/850
141/850
144/850
144/850
147/850
147/850
150/850
150/850
153/850
153/850
156/850
156/850
159/850
159/850
162/850
162/850
165/850
165/850
168/850
168/850
171/850
171/850
174/850
174/850


In [12]:
tifffile.imwrite(save_directory + '/output.tif',fusion_stack)

img = xz_img.transpose(1,0,2)
tifffile.imwrite(save_directory + '/input.tif',img)

### 3.4 Napari Visualization

In [11]:
os.environ.pop("QT_PLUGIN_PATH", None)
os.environ["QT_QPA_PLATFORM"] = "wayland"

viewer = napari.Viewer()
viewer.add_image(img, name='input')
viewer.add_image(fusion_stack, name='output')
napari.run()